In [ ]:
#!/usr/bin/env python
"""
MultiGrate + Cox survival training pipeline for TCGA-BRCA.

This mirrors the TEA-seq train_multigrate_split_3.ipynb pattern:
  - train MultiGrate on the frozen training split only
  - extract train embeddings
  - fit a downstream linear Cox head on the embeddings
  - save the MultiVAE model, embedding scaler, and Cox coefficients

Run generate_brca_survival_splits.ipynb and preprocess_cnv_brca.ipynb first.
"""

from __future__ import annotations
from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import multigrate as mtg
import scvi
import torch
import torch.nn as nn
from torch.optim import AdamW
from sklearn.preprocessing import StandardScaler
import joblib


In [ ]:
# -----------------------------
# Config
# -----------------------------

matrices_dir = Path("../matrices")
data_dir     = Path("../data")
splits_dir   = Path("../splits")
model_dir    = Path("../models/multigrate_survival")
results_dir  = Path("../results/brca_survival/multigrate_train")

model_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

SPLIT_TAG = "brca_survival"
Z_DIM = 70
MAX_EPOCHS_MULTIGRATE = 200
COX_EPOCHS = 2000
COX_LR = 1e-3
COX_WEIGHT_DECAY = 1e-4
SEED = 0

model_root = model_dir / f"{SPLIT_TAG}_multigrate_cox"
vae_dir = model_root / "multivae_model"
scaler_path = model_root / "embedding_scaler.pkl"
cox_coef_path = model_root / "cox_coefficients.csv"
metadata_path = model_root / "metadata.json"
model_root.mkdir(parents=True, exist_ok=True)

print("Model root:", model_root.resolve())
print("Results dir:", results_dir.resolve())


In [ ]:
# -----------------------------
# Data utilities
# -----------------------------

def load_index_csv_0based(path: Path, n_total: int) -> np.ndarray:
    df = pd.read_csv(path)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) == 0:
        raise ValueError(f"No numeric index column in {path}")
    idx = df[num_cols[0]].to_numpy()
    if np.isnan(idx).any():
        raise ValueError(f"NaN index in {path}")
    idx_int = idx.astype(np.int64)
    if not np.allclose(idx, idx_int):
        raise ValueError(f"Non-integer index in {path}")

    mn, mx = int(idx_int.min()), int(idx_int.max())
    if mn == 0 and mx == n_total - 1:
        idx0 = idx_int
    elif mn == 1 and mx == n_total:
        idx0 = idx_int - 1
    else:
        idx0 = idx_int

    if (idx0 < 0).any() or (idx0 >= n_total).any():
        raise ValueError(f"Out-of-range index in {path}: [{idx0.min()}, {idx0.max()}], n={n_total}")
    return idx0


def matrix_to_anndata(X: np.ndarray, obs_names: list[str], var_names: list[str]) -> ad.AnnData:
    A = ad.AnnData(X=X.astype(np.float32))
    A.obs_names = pd.Index(obs_names).astype(str)
    A.var_names = pd.Index(var_names).astype(str)
    A.layers["norm"] = A.X.copy()
    return A


def load_brca_data():
    sample_ids = pd.read_csv(splits_dir / f"{SPLIT_TAG}_sample_ids.csv")["sample_id"].tolist()
    idx_tr = load_index_csv_0based(splits_dir / f"{SPLIT_TAG}_train_idx.csv", len(sample_ids))
    idx_te = load_index_csv_0based(splits_dir / f"{SPLIT_TAG}_test_idx.csv", len(sample_ids))

    print("Loading matrices...")
    rna_df  = pd.read_csv(matrices_dir / "RNA_X_full.csv", index_col=0)
    meth_df = pd.read_csv(matrices_dir / "Methylation_X_full.csv", index_col=0)
    cnv_df  = pd.read_csv(matrices_dir / "CNV_X_ld_features.csv", index_col=0)

    surv_raw = pd.read_csv(data_dir / "BRCA_survival.tsv", sep="\t", index_col=0)
    surv_df = surv_raw[["OS", "OS.time"]].copy()
    surv_df.columns = ["event", "time"]
    surv_df["event"] = pd.to_numeric(surv_df["event"], errors="coerce")
    surv_df["time"] = pd.to_numeric(surv_df["time"], errors="coerce")
    surv_df = surv_df.dropna(subset=["event", "time"])
    surv_df = surv_df[surv_df["time"] > 0]

    for name, df in [("RNA", rna_df), ("Methylation", meth_df), ("CNV", cnv_df), ("survival", surv_df)]:
        missing = set(sample_ids) - set(df.index)
        if missing:
            raise ValueError(f"{name} missing sample {next(iter(missing))}")

    X_rna = rna_df.loc[sample_ids].to_numpy(dtype=np.float32)
    X_meth = meth_df.loc[sample_ids].to_numpy(dtype=np.float32)
    X_cnv = cnv_df.loc[sample_ids].to_numpy(dtype=np.float32)
    event = surv_df.loc[sample_ids, "event"].to_numpy(dtype=np.float32)
    time = surv_df.loc[sample_ids, "time"].to_numpy(dtype=np.float32)

    if not all(np.isfinite(x).all() for x in [X_rna, X_meth, X_cnv, event, time]):
        raise ValueError("Non-finite value found in loaded data")
    if (time <= 0).any():
        raise ValueError("Non-positive survival time found")

    return sample_ids, idx_tr, idx_te, X_rna, X_meth, X_cnv, event, time, rna_df.columns, meth_df.columns, cnv_df.columns


def build_multigrate_adata(rna: ad.AnnData, meth: ad.AnnData, cnv: ad.AnnData):
    meth.X = meth.layers["norm"].astype(np.float32)
    cnv.X = cnv.layers["norm"].astype(np.float32)

    adatas = [[rna], [meth], [cnv]]
    adata = mtg.data.organize_multimodal_anndatas(
        adatas=adatas,
        layers=[["norm"], ["norm"], ["norm"]],
    )
    return adata, rna.shape[1]


In [ ]:
# -----------------------------
# Cox utilities
# -----------------------------

def cox_partial_loss(log_hazard: torch.Tensor, time: torch.Tensor, event: torch.Tensor) -> torch.Tensor:
    order = torch.argsort(time, descending=True)
    lh = log_hazard[order]
    ev = event[order]
    log_cumsum = torch.logcumsumexp(lh, dim=0)
    n_events = ev.sum().clamp_min(1.0)
    return -((lh - log_cumsum) * ev).sum() / n_events


def concordance_index_survival(time: np.ndarray, event: np.ndarray, log_risk: np.ndarray) -> float:
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    score = -np.asarray(log_risk, dtype=float)
    concordant = 0.0
    permissible = 0.0
    n = len(time)
    for i in range(n):
        for j in range(i + 1, n):
            if time[i] == time[j]:
                continue
            if time[i] < time[j] and event[i] == 1:
                permissible += 1.0
                if score[i] < score[j]:
                    concordant += 1.0
                elif score[i] == score[j]:
                    concordant += 0.5
            elif time[j] < time[i] and event[j] == 1:
                permissible += 1.0
                if score[j] < score[i]:
                    concordant += 1.0
                elif score[j] == score[i]:
                    concordant += 0.5
    return float(concordant / permissible) if permissible else float("nan")


def fit_linear_cox(Z: np.ndarray, event: np.ndarray, time: np.ndarray, *, epochs: int, lr: float, weight_decay: float, seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    Z_t = torch.tensor(Z, dtype=torch.float32, device=device)
    event_t = torch.tensor(event, dtype=torch.float32, device=device)
    time_t = torch.tensor(time, dtype=torch.float32, device=device)

    model = nn.Linear(Z.shape[1], 1, bias=False).to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best = {"loss": float("inf"), "coef": None}

    for ep in range(epochs):
        opt.zero_grad()
        log_risk = model(Z_t).squeeze(1)
        loss = cox_partial_loss(log_risk, time_t, event_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        val = float(loss.detach().cpu())
        if val < best["loss"]:
            best["loss"] = val
            best["coef"] = model.weight.detach().cpu().numpy().reshape(-1).copy()
        if ep % 200 == 0:
            print(f"[Cox] epoch {ep:4d} | loss {val:.6f}")

    beta = best["coef"]
    log_risk = Z @ beta
    return beta, log_risk, best["loss"]


In [ ]:
# -----------------------------
# Main training function
# -----------------------------

def train_multigrate_brca_survival():
    scvi.settings.seed = SEED
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    print("[Step 1] Loading BRCA data...")
    sample_ids, idx_tr, idx_te, X_rna, X_meth, X_cnv, event, time, rna_cols, meth_cols, cnv_cols = load_brca_data()
    train_ids = [sample_ids[i] for i in idx_tr]

    print(f"Train: {len(idx_tr)} samples | events: {int(event[idx_tr].sum())}")
    print(f"Test : {len(idx_te)} samples | events: {int(event[idx_te].sum())}")

    rna_tr = matrix_to_anndata(X_rna[idx_tr], train_ids, list(rna_cols))
    meth_tr = matrix_to_anndata(X_meth[idx_tr], train_ids, list(meth_cols))
    cnv_tr = matrix_to_anndata(X_cnv[idx_tr], train_ids, list(cnv_cols))

    print("[Step 2] Training MultiGrate...")
    adata_tr, rna_end = build_multigrate_adata(rna_tr, meth_tr, cnv_tr)
    mtg.model.MultiVAE.setup_anndata(adata_tr, rna_indices_end=rna_end)
    vae = mtg.model.MultiVAE(adata_tr, losses=["mse", "mse", "mse"], z_dim=Z_DIM)
    vae.train(max_epochs=MAX_EPOCHS_MULTIGRATE)

    print("[Step 3] Extracting train embeddings...")
    vae.get_model_output()
    if "X_multigrate" not in adata_tr.obsm:
        raise RuntimeError("Missing adata_tr.obsm['X_multigrate'] after get_model_output()")
    Z_train = np.asarray(adata_tr.obsm["X_multigrate"], dtype=np.float32)

    print("[Step 4] Standardizing embeddings and fitting Cox head...")
    scaler = StandardScaler()
    Z_train_s = scaler.fit_transform(Z_train).astype(np.float32)
    beta, log_risk_train, cox_loss = fit_linear_cox(
        Z_train_s, event[idx_tr], time[idx_tr],
        epochs=COX_EPOCHS, lr=COX_LR, weight_decay=COX_WEIGHT_DECAY, seed=SEED,
    )
    c_train = concordance_index_survival(time[idx_tr], event[idx_tr], log_risk_train)
    print(f"Train C-index: {c_train:.4f}")

    print("[Step 5] Saving artifacts...")
    vae.save(str(vae_dir), overwrite=True)
    joblib.dump(scaler, scaler_path)
    pd.DataFrame({"factor": [f"F{i+1}" for i in range(len(beta))], "coef": beta}).to_csv(cox_coef_path, index=False)

    np.save(results_dir / "train_embedding.npy", Z_train.astype(np.float32))
    np.save(results_dir / "train_embedding_scaled.npy", Z_train_s.astype(np.float32))
    np.save(results_dir / "log_risk_train.npy", log_risk_train.astype(np.float32))

    metadata = {
        "method": "multigrate+linear_cox",
        "task": "survival",
        "split_tag": SPLIT_TAG,
        "z_dim": Z_DIM,
        "max_epochs_multigrate": MAX_EPOCHS_MULTIGRATE,
        "cox_epochs": COX_EPOCHS,
        "cox_lr": COX_LR,
        "cox_weight_decay": COX_WEIGHT_DECAY,
        "cox_train_loss": float(cox_loss),
        "c_index_train": float(c_train),
        "n_train": int(len(idx_tr)),
        "n_events_train": int(event[idx_tr].sum()),
        "n_test": int(len(idx_te)),
        "n_events_test": int(event[idx_te].sum()),
        "vae_dir": str(vae_dir),
        "scaler": str(scaler_path),
        "cox_coefficients": str(cox_coef_path),
        "views": ["RNA", "Methylation", "CNV"],
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)
    with open(results_dir / "metrics_train.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print("Saved model artifacts under:", model_root.resolve())
    print(json.dumps(metadata, indent=2))


if __name__ == "__main__":
    train_multigrate_brca_survival()
